In [1]:
import pandas as pd
import numpy as np
import oracledb
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.preprocessing import StandardScaler

In [2]:
conn = oracledb.connect(user='pfe', password='pfe', dsn='127.0.0.1:1521/XEPDB1')
df = pd.read_sql('SELECT * FROM dataset_final1', conn)
conn.close()

C:\Users\fatimazahra\AppData\Local\Temp\ipykernel_9648\752863821.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('SELECT * FROM dataset_final1', conn)


In [3]:
df = df.drop(columns=[
    'LIBELLE_SECTEUR','LIBELLE_REGION','LIBELLE_VILLE',
    'TARGET_RECOUVREMENT','TARGET_POURCENTAGE_IMPAYE',
    'FLAG_GLOBAL','TARGET_RECOUVREMENT_NEW1',
    'TARGET_RECOUVREMENT_NEW2','TARGET_FLAG_CLEAN'
])

In [4]:
y = df["TARGET_POURCENTAGE_NEW1"]

X = df.drop(columns=["TARGET_POURCENTAGE_NEW1"])

In [5]:
# ================================
#  6. SPLIT PAR AFFILIÉS
# ================================
# On évite le data leakage en séparant par NUM_AFF

affilies = df["NUM_AFF"].unique()

# Mélange pour éviter biais
np.random.seed(42)
np.random.shuffle(affilies)

# Split fixe
train_aff = affilies[:130000]
test_aff  = affilies[130000:]

# Construction datasets
train_df = df[df["NUM_AFF"].isin(train_aff)].copy()
test_df  = df[df["NUM_AFF"].isin(test_aff)].copy()

print("Train affiliés:", train_df["NUM_AFF"].nunique())
print("Test affiliés:", test_df["NUM_AFF"].nunique())

# ================================
#  7. FEATURES / TARGET SPLIT
# ================================
y_train = train_df["TARGET_POURCENTAGE_NEW1"]
X_train = train_df.drop(columns=["TARGET_POURCENTAGE_NEW1"])

y_test = test_df["TARGET_POURCENTAGE_NEW1"]
X_test = test_df.drop(columns=["TARGET_POURCENTAGE_NEW1"])

Train affiliés: 130000
Test affiliés: 22834


In [6]:
X_train = X_train.select_dtypes(include=['int64','float64'])
X_test = X_test.select_dtypes(include=['int64','float64'])

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
#gestion de desiquilibre
# poids automatique
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Scale_pos_weight:", scale_pos_weight)

Scale_pos_weight: inf


C:\Users\fatimazahra\AppData\Local\Temp\ipykernel_9648\1608815413.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()


In [9]:
from xgboost import XGBRegressor

def xgb_objective(trial):

    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 200, 600),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_test_scaled)

    return r2_score(y_test, preds)

In [10]:
from lightgbm import LGBMRegressor

def lgb_objective(trial):

    model = LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 200, 600),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
        num_leaves=trial.suggest_int("num_leaves", 20, 80),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_test_scaled)

    return r2_score(y_test, preds)

In [11]:
def rf_objective(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 150),
        max_depth=trial.suggest_int("max_depth", 5, 12),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,  # utilise tous les CPU
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    return r2_score(y_test, preds)

In [12]:
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=30)

[I 2026-04-30 11:11:49,293] A new study created in memory with name: no-name-b07c2bda-7e45-42eb-ad5e-73ae6c467061
[I 2026-04-30 11:12:09,285] Trial 0 finished with value: 0.8764824539672604 and parameters: {'n_estimators': 598, 'max_depth': 4, 'learning_rate': 0.10746051852991674, 'subsample': 0.8313276038033843, 'colsample_bytree': 0.7067792960140412}. Best is trial 0 with value: 0.8764824539672604.
[I 2026-04-30 11:12:56,371] Trial 1 finished with value: 0.8632134336476591 and parameters: {'n_estimators': 477, 'max_depth': 10, 'learning_rate': 0.14031258125095294, 'subsample': 0.916370813210309, 'colsample_bytree': 0.9999419206378376}. Best is trial 0 with value: 0.8764824539672604.
[I 2026-04-30 11:13:28,545] Trial 2 finished with value: 0.8640602883991855 and parameters: {'n_estimators': 478, 'max_depth': 8, 'learning_rate': 0.13417718020361702, 'subsample': 0.9124912872442726, 'colsample_bytree': 0.9301088591880124}. Best is trial 0 with value: 0.8764824539672604.
[I 2026-04-30 11

In [13]:
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_objective, n_trials=30)

[I 2026-04-30 11:20:24,564] A new study created in memory with name: no-name-637faf3f-11bf-40b9-a951-ee5efca81ab8


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.215929 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4363
[LightGBM] [Info] Number of data points in the train set: 130000, number of used features: 26
[LightGBM] [Info] Start training from score 13.189193
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

c:\Users\fatimazahra\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-04-30 11:20:57,776] Trial 0 finished with value: 0.8716444479226354 and parameters: {'n_estimators': 477, 'learning_rate': 0.183254102309641, 'num_leaves': 28, 'max_depth': 6}. Best is trial 0 with value: 0.8716444479226354.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.088648 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4363
[LightGBM] [Info] Number of data points in the train set: 130000, number of used features: 26
[LightGBM] [Info] Start training from score 13.189193
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\Users\fatimazahra\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-04-30 11:21:09,613] Trial 1 finished with value: 0.8766206516605143 and parameters: {'n_estimators': 261, 'learning_rate': 0.06979504516570204, 'num_leaves': 46, 'max_depth': 8}. Best is trial 1 with value: 0.8766206516605143.


...

c:\Users\fatimazahra\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-04-30 11:26:22,326] Trial 27 finished with value: 0.87727247943711 and parameters: {'n_estimators': 458, 'learning_rate': 0.055811001799953755, 'num_leaves': 33, 'max_depth': 11}. Best is trial 25 with value: 0.8781459423414137.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016071 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4363
[LightGBM] [Info] Number of data points in the train set: 130000, number of used features: 26
[LightGBM] [Info] Start training from score 13.189193


c:\Users\fatimazahra\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-04-30 11:26:33,403] Trial 28 finished with value: 0.8779300866909159 and parameters: {'n_estimators': 376, 'learning_rate': 0.028469905157321927, 'num_leaves': 43, 'max_depth': 13}. Best is trial 25 with value: 0.8781459423414137.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.065870 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4363
[LightGBM] [Info] Number of data points in the train set: 130000, number of used features: 26
[LightGBM] [Info] Start training from score 13.189193


c:\Users\fatimazahra\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-04-30 11:26:44,002] Trial 29 finished with value: 0.8775049941076603 and parameters: {'n_estimators': 479, 'learning_rate': 0.04692673401723929, 'num_leaves': 28, 'max_depth': 9}. Best is trial 25 with value: 0.8781459423414137.


In [14]:
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(rf_objective, n_trials=30)

[I 2026-04-30 11:26:44,069] A new study created in memory with name: no-name-9904b3cd-c302-4ddf-9f7d-421525e0f18a
[I 2026-04-30 11:27:00,850] Trial 0 finished with value: 0.8767463164932515 and parameters: {'n_estimators': 74, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.8767463164932515.
[I 2026-04-30 11:27:13,931] Trial 1 finished with value: 0.8758703615808445 and parameters: {'n_estimators': 82, 'max_depth': 7, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.8767463164932515.
[I 2026-04-30 11:27:29,925] Trial 2 finished with value: 0.8735203233541846 and parameters: {'n_estimators': 138, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.8767463164932515.
[I 2026-04-30 11:27:53,873] Trial 3 finished with value: 0.877258531567862 and parameters: {'n_estimators': 96, 'max_depth': 10, 'min_sam

In [15]:
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_xgb = XGBRegressor(**study_xgb.best_params, random_state=42)
best_lgb = LGBMRegressor(**study_lgb.best_params, random_state=42)
best_rf.fit(X_train_scaled, y_train)
best_xgb.fit(X_train_scaled, y_train)
best_lgb.fit(X_train_scaled, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058294 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4363
[LightGBM] [Info] Number of data points in the train set: 130000, number of used features: 26
[LightGBM] [Info] Start training from score 13.189193


,boosting_type,'gbdt'
,num_leaves,42
,max_depth,12
,learning_rate,0.010079020432770097
,n_estimators,471
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(y_true, y_pred, name):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print("\n=====", name, "=====")
    print("MAE  :", round(mae, 4))
    print("RMSE :", round(rmse, 4))
    print("R2   :", round(r2, 4))
   

    return {
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
      
    }

In [20]:
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_rf.fit(X_train, y_train)

y_pred_rf = best_rf.predict(X_test)

res_rf = evaluate_model(y_test, y_pred_rf, "Random Forest")

best_lgb = LGBMRegressor(**study_lgb.best_params, random_state=42)
best_lgb.fit(X_train, y_train)

y_pred_lgb = best_lgb.predict(X_test)

res_lgb = evaluate_model(y_test, y_pred_lgb, "LightGBM")

best_xgb = XGBRegressor(**study_xgb.best_params, random_state=42)
best_xgb.fit(X_train, y_train)

y_pred_xgb = best_xgb.predict(X_test)

res_xgb = evaluate_model(y_test, y_pred_xgb, "XGBoost")


===== Random Forest =====
MAE  : 3.8797
RMSE : 8.1065
R2   : 0.8773
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.077811 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4357
[LightGBM] [Info] Number of data points in the train set: 130000, number of used features: 26
[LightGBM] [Info] Start training from score 13.189193

===== LightGBM =====
MAE  : 3.9365
RMSE : 8.0812
R2   : 0.8781

===== XGBoost =====
MAE  : 3.8616
RMSE : 8.069
R2   : 0.8785


In [21]:
import pandas as pd

results = pd.DataFrame([res_rf, res_lgb, res_xgb])

print("\n===== COMPARAISON DES MODELES =====")
print(results.sort_values("R2", ascending=False))
"""

===== COMPARAISON DES MODELES =====
           model       MAE      RMSE        R2          MAPE
2        XGBoost  3.863009  8.070102  0.878421  4.186083e+10
1       LightGBM  3.859956  8.071533  0.878378  4.146546e+10
0  Random Forest  3.886506  8.105945  0.877338  4.281393e+10
"""


===== COMPARAISON DES MODELES =====
           model       MAE      RMSE        R2
2        XGBoost  3.861635  8.069026  0.878453
1       LightGBM  3.936524  8.081247  0.878085
0  Random Forest  3.879738  8.106547  0.877320


'\n\n===== COMPARAISON DES MODELES =====\n           model       MAE      RMSE        R2          MAPE\n2        XGBoost  3.863009  8.070102  0.878421  4.186083e+10\n1       LightGBM  3.859956  8.071533  0.878378  4.146546e+10\n0  Random Forest  3.886506  8.105945  0.877338  4.281393e+10\n'